# Campaign Instance Detection + LLM Embeddings + RAG

This notebook loads the campaign dataset from `06_method_b_pagination_and_records.json`, detects different campaign instances, and computes text embeddings with Hugging Face BERT-mini (`prajjwal1/bert-mini`).

## 1) Install Dependencies (If Needed)

This cell installs all packages used in the notebook: data processing, embeddings, and multi-provider LLM access for the RAG answer step. In Google Colab, run it once per runtime.

In [ ]:
# If needed, uncomment and run this cell to install dependencies.
# In Colab: run once per session, then restart runtime if prompted.
# %pip install -q pandas numpy torch transformers tqdm python-dotenv requests openai anthropic google-generativeai plotly umap-learn

## 2) Import Libraries

These imports cover: JSON/dataframe handling, BERT-mini embeddings, similarity search, environment variable loading, and API calls for multiple LLM providers.

In [2]:
import json
import os
import re
from pathlib import Path
from urllib.parse import urlparse

import numpy as np
import pandas as pd
import requests
import torch
from dotenv import load_dotenv
from tqdm.auto import tqdm
from transformers import AutoModel, AutoTokenizer

## 3) Load the Campaign Records

This cell reads the JSON file and creates a dataframe with one row per campaign record.

In [17]:
DATA_PATH = Path("06_method_b_pagination_and_records.json")

with DATA_PATH.open("r", encoding="utf-8") as f:
    raw = json.load(f)

records = raw.get("records", [])
print(f"Records loaded: {len(records)}")

df = pd.DataFrame(records)
df.to_csv("campaigns_parsed.csv", index=False)
df.head()

Records loaded: 57


,url,title,preview,body_excerpt,paragraph_count,error
0,https://www.swiss-solidarity.org/campaigns/aid...,Aid following the earthquake in Nepal,"We need you Your donation, our action.","We need you Your donation, our action. Get inv...",46.0,NaN
1,https://www.swiss-solidarity.org/campaigns/aid...,Aid for Afghanistan,"We need you Your donation, our action.","We need you Your donation, our action. Get inv...",35.0,NaN
2,https://www.swiss-solidarity.org/campaigns/aid...,Aid in response to disasters and crises in Swi...,"We need you Your donation, our action.","We need you Your donation, our action. Get inv...",60.0,NaN
3,https://www.swiss-solidarity.org/campaigns/asi...,Asia 2009,"We need you Your donation, our action.","We need you Your donation, our action. Get inv...",32.0,NaN
4,https://www.swiss-solidarity.org/campaigns/chi...,Child Protection in Switzerland,"We need you Your donation, our action.","We need you Your donation, our action. Get inv...",40.0,NaN


## 4) Detect Campaign Instances

Campaigns can appear as repeated variants (for example URL suffixes like `-1` or year suffixes). This step builds a campaign family key and assigns an instance number for each row.

In [4]:
def slug_from_url(url: str) -> str:
    path = urlparse(url).path.strip("/")
    return path.split("/")[-1] if path else ""

def normalize_title(title: str) -> str:
    title = title.lower().strip()
    title = re.sub(r"[^a-z0-9\s]+", " ", title)
    title = re.sub(r"\s+", " ", title).strip()
    return title

def base_slug(slug: str) -> str:
    # Treat trailing numeric suffixes (e.g. -1, -2010) as possible instance markers.
    return re.sub(r"-\d+$", "", slug)

df["slug"] = df["url"].map(slug_from_url)
df["slug_base"] = df["slug"].map(base_slug)
df["title_norm"] = df["title"].fillna("").map(normalize_title)

# Campaign family key used to detect different instances of the same campaign concept.
df["campaign_family_key"] = df["title_norm"] + "||" + df["slug_base"]

df = df.sort_values(["campaign_family_key", "url"]).reset_index(drop=True)
df["instance_number"] = df.groupby("campaign_family_key").cumcount() + 1
df["instance_count_in_family"] = df.groupby("campaign_family_key")["campaign_family_key"].transform("size")

df["campaign_instance_id"] = np.where(
    df["instance_count_in_family"] > 1,
    df["campaign_family_key"] + "__instance_" + df["instance_number"].astype(str),
    df["campaign_family_key"] + "__instance_1",
)

duplicates = df[df["instance_count_in_family"] > 1][
    ["title", "url", "campaign_family_key", "instance_number", "instance_count_in_family"]
]

print("Campaign families:", df["campaign_family_key"].nunique())
print("Detected multi-instance rows:", len(duplicates))
duplicates.head(20)

Campaign families: 55
Detected multi-instance rows: 4


,title,url,campaign_family_key,instance_number,instance_count_in_family
21,Flooding in Pakistan,https://www.swiss-solidarity.org/campaigns/flo...,flooding in pakistan||flooding-in-pakistan,1,2
22,Flooding in Pakistan,https://www.swiss-solidarity.org/campaigns/flo...,flooding in pakistan||flooding-in-pakistan,2,2
29,Hunger in East Africa,https://www.swiss-solidarity.org/campaigns/hun...,hunger in east africa||hunger-in-east-africa,1,2
30,Hunger in East Africa,https://www.swiss-solidarity.org/campaigns/hun...,hunger in east africa||hunger-in-east-africa,2,2


## 5) Compute BERT-mini Embeddings

Here we create one text per campaign (`title + preview + body_excerpt`) and embed it with `prajjwal1/bert-mini`. We mean-pool token embeddings and normalize vectors for cosine similarity search.

In [5]:
# Build a single text field per campaign for embedding.
def combine_text(row: pd.Series) -> str:
    parts = [
        str(row.get("title", "") or ""),
        str(row.get("preview", "") or ""),
        str(row.get("body_excerpt", "") or ""),
    ]
    return " ".join([p.strip() for p in parts if p and p.strip()])

df["campaign_text"] = df.apply(combine_text, axis=1)

EMBEDDING_MODEL_NAME = "prajjwal1/bert-mini"
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

tokenizer = AutoTokenizer.from_pretrained(EMBEDDING_MODEL_NAME)
embedding_model = AutoModel.from_pretrained(EMBEDDING_MODEL_NAME).to(device)
embedding_model.eval()

def mean_pool(last_hidden_state: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
    mask = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
    masked_embeddings = last_hidden_state * mask
    summed = masked_embeddings.sum(dim=1)
    counts = torch.clamp(mask.sum(dim=1), min=1e-9)
    return summed / counts

def embed_texts(texts, batch_size: int = 16, max_length: int = 256):
    all_vecs = []
    for i in tqdm(range(0, len(texts), batch_size), desc="Embedding"):
        batch = texts[i:i + batch_size]
        encoded = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt",
        )
        encoded = {k: v.to(device) for k, v in encoded.items()}

        with torch.no_grad():
            outputs = embedding_model(**encoded)
            vecs = mean_pool(outputs.last_hidden_state, encoded["attention_mask"])
            vecs = torch.nn.functional.normalize(vecs, p=2, dim=1)

        all_vecs.append(vecs.cpu().numpy())

    return np.vstack(all_vecs)

def embed_one_text(text: str, max_length: int = 256) -> np.ndarray:
    vec = embed_texts([text], batch_size=1, max_length=max_length)
    return vec[0]

embeddings = embed_texts(df["campaign_text"].tolist(), batch_size=16, max_length=256)

print("Embeddings shape:", embeddings.shape)
print("Expected rows == records:", embeddings.shape[0] == len(df))

df["embedding_dim"] = embeddings.shape[1]
df[["title", "campaign_instance_id", "embedding_dim"]].head()

Using device: cpu


config.json:   0%|          | 0.00/286 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/45.1M [00:00<?, ?B/s]

Embedding:   0%|          | 0/4 [00:00<?, ?it/s]

Embeddings shape: (57, 256)
Expected rows == records: True


,title,campaign_instance_id,embedding_dim
0,Aid following the earthquake in Nepal,aid following the earthquake in nepal||aid-fol...,256
1,Aid for Afghanistan,aid for afghanistan||aid-for-afghanistan__inst...,256
2,Aid in response to disasters and crises in Swi...,aid in response to disasters and crises in swi...,256
3,Asia 2009,asia 2009||asia__instance_1,256
4,Child Protection in Switzerland,child protection in switzerland||child-protect...,256


model.safetensors:   0%|          | 0.00/45.1M [00:00<?, ?B/s]

## 6) Save Embeddings and Metadata

This exports campaign metadata and vectors so you can reuse them without recomputing embeddings each time.

In [6]:
# Persist outputs for downstream retrieval / RAG steps.
output_table = df[[
    "url",
    "title",
    "campaign_instance_id",
    "campaign_family_key",
    "instance_number",
    "instance_count_in_family",
    "campaign_text",
]].copy()

# 1) Existing exports
output_table.to_csv("campaign_instances.csv", index=False)
np.save("campaign_embeddings_bert_mini.npy", embeddings)

# 2) Universal RAG export: JSONL with one record per document + embedding vector
rag_export = output_table.copy()
rag_export["embedding"] = [vec.tolist() for vec in embeddings]
rag_export["id"] = rag_export["campaign_instance_id"]
rag_export["text"] = rag_export["campaign_text"]

rag_export = rag_export[[
    "id",
    "text",
    "title",
    "url",
    "campaign_family_key",
    "instance_number",
    "instance_count_in_family",
    "embedding",
]]

rag_export.to_json(
    "campaign_embeddings_rag.jsonl",
    orient="records",
    lines=True,
    force_ascii=False,
)

# 3) Optional Parquet export (widely used by vector DB ingestion pipelines)
try:
    rag_export.to_parquet("campaign_embeddings_rag.parquet", index=False)
    parquet_status = "Saved: campaign_embeddings_rag.parquet"
except Exception as e:
    parquet_status = f"Parquet not saved ({type(e).__name__}: {e})"

print("Saved: campaign_instances.csv")
print("Saved: campaign_embeddings_bert_mini.npy")
print("Saved: campaign_embeddings_rag.jsonl")
print(parquet_status)
rag_export.head()

Saved: campaign_instances.csv
Saved: campaign_embeddings_bert_mini.npy
Saved: campaign_embeddings_rag.jsonl
Saved: campaign_embeddings_rag.parquet


,id,text,title,url,campaign_family_key,instance_number,instance_count_in_family,embedding
0,aid following the earthquake in nepal||aid-fol...,Aid following the earthquake in Nepal We need ...,Aid following the earthquake in Nepal,https://www.swiss-solidarity.org/campaigns/aid...,aid following the earthquake in nepal||aid-fol...,1,1,"[-0.062477853149175644, 0.11125241219997406, 0..."
1,aid for afghanistan||aid-for-afghanistan__inst...,"Aid for Afghanistan We need you Your donation,...",Aid for Afghanistan,https://www.swiss-solidarity.org/campaigns/aid...,aid for afghanistan||aid-for-afghanistan,1,1,"[-0.06372381001710892, 0.10828344523906708, 0...."
2,aid in response to disasters and crises in swi...,Aid in response to disasters and crises in Swi...,Aid in response to disasters and crises in Swi...,https://www.swiss-solidarity.org/campaigns/aid...,aid in response to disasters and crises in swi...,1,1,"[-0.06566299498081207, 0.11273548007011414, 0...."
3,asia 2009||asia__instance_1,"Asia 2009 We need you Your donation, our actio...",Asia 2009,https://www.swiss-solidarity.org/campaigns/asi...,asia 2009||asia,1,1,"[-0.0665421411395073, 0.10772591084241867, 0.0..."
4,child protection in switzerland||child-protect...,Child Protection in Switzerland We need you Yo...,Child Protection in Switzerland,https://www.swiss-solidarity.org/campaigns/chi...,child protection in switzerland||child-protect...,1,1,"[-0.06481823325157166, 0.11041181534528732, 0...."


## 7) Interactive Embedding Map (Plotly)

This section creates an interactive 2D map of campaign embeddings so you can visually inspect semantic similarity between campaigns.

What the code does:
- Uses the embedding matrix (`embeddings`) and projects high-dimensional vectors to 2 dimensions with UMAP.
- Builds a plotting table with campaign metadata (`title`, `url`, `campaign_instance_id`, `campaign_family_key`).
- Creates a Plotly scatter chart where each point is one campaign.

How to read and use the plot:
- Hover on a point to see the campaign title (plus URL and IDs in the tooltip).
- Points that are closer together in 2D are generally more semantically similar in embedding space.
- The axes correspond to UMAP Dimension 1 and UMAP Dimension 2.

Implementation note: if campaign instance columns are missing in the current kernel session, the code reconstructs them automatically before plotting, so the cell works independently.

In [20]:
import plotly.express as px
import umap.umap_ as umap

# Ensure required metadata columns exist even if Step 4 was not run in this session.
plot_df = df.copy()
if "campaign_family_key" not in plot_df.columns or "campaign_instance_id" not in plot_df.columns:
    local_slug = plot_df["url"].fillna("").str.rstrip("/").str.split("/").str[-1]
    local_slug_base = local_slug.str.replace(r"-\d+$", "", regex=True)
    local_title_norm = (
        plot_df["title"]
        .fillna("")
        .str.lower()
        .str.replace(r"[^a-z0-9\s]+", " ", regex=True)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )
    plot_df["campaign_family_key"] = local_title_norm + "||" + local_slug_base
    plot_df["instance_number"] = plot_df.groupby("campaign_family_key").cumcount() + 1
    plot_df["campaign_instance_id"] = (
        plot_df["campaign_family_key"] + "__instance_" + plot_df["instance_number"].astype(str)
    )

# Project embeddings (N x D) to 2D with UMAP for visualization.
X = embeddings.astype(np.float32, copy=False)
umap_model = umap.UMAP(n_components=2, random_state=42)
coords_2d = umap_model.fit_transform(X)

plot_df = plot_df[["title", "url", "campaign_instance_id", "campaign_family_key"]].copy()
plot_df["x"] = coords_2d[:, 0]
plot_df["y"] = coords_2d[:, 1]

fig = px.scatter(
    plot_df,
    x="x",
    y="y",
    hover_name="title",
    hover_data={
        "url": True,
        "campaign_instance_id": True,
        "campaign_family_key": True,
        "x": False,
        "y": False,
    },
    title="Campaign Embeddings (UMAP 2D projection)",
    opacity=0.85,
    height=650,
    template="plotly_white",
)

fig.update_traces(marker={"size": 9, "line": {"width": 0.5, "color": "#1f2937"}})
fig.update_layout(
    xaxis_title="UMAP Dimension 1",
    yaxis_title="UMAP Dimension 2",
    hoverlabel={"namelength": -1},
)

fig.show()

/opt/anaconda3/envs/py_3_13_2/lib/python3.13/site-packages/sklearn/utils/deprecation.py:151: FutureWarning:

'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.

/opt/anaconda3/envs/py_3_13_2/lib/python3.13/site-packages/umap/umap_.py:1952: UserWarning:

n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.

OMP: Info #276: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.


## 8) Load Secrets from `.env` or Google Colab

This section loads API keys from a local `.env` file and falls back to Google Colab Secrets when running in Colab. That way the same notebook can run locally or in Colab with minimal changes.

In [7]:
load_dotenv(override=False)

def _read_colab_secret(secret_name: str):
    try:
        from google.colab import userdata  # type: ignore
    except Exception:
        return None

    try:
        value = userdata.get(secret_name)
        return value if value else None
    except Exception:
        return None

def get_secret(secret_name: str, default: str = "") -> str:
    # Priority: local env > Colab secret > default.
    env_val = os.getenv(secret_name)
    if env_val:
        return env_val
    colab_val = _read_colab_secret(secret_name)
    if colab_val:
        return colab_val
    return default

API_KEYS = {
    "OPENAI_API_KEY": get_secret("OPENAI_API_KEY"),
    "GEMINI_API_KEY": get_secret("GEMINI_API_KEY"),
    "ANTHROPIC_API_KEY": get_secret("ANTHROPIC_API_KEY"),
    "HUGGINGFACE_API_KEY": get_secret("HUGGINGFACE_API_KEY"),
}

for key_name, key_value in API_KEYS.items():
    print(f"{key_name}: {'SET' if key_value else 'MISSING'}")

LLM_PROVIDER = os.getenv("LLM_PROVIDER", "gemini_model").strip().lower()
OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-4o-mini")
GEMINI_MODEL = os.getenv("GEMINI_MODEL", "gemma-4-31b-it")
ANTHROPIC_MODEL = os.getenv("ANTHROPIC_MODEL", "claude-3-5-haiku-latest")
HUGGINGFACE_MODEL = os.getenv("HUGGINGFACE_MODEL", "google/flan-t5-base")

TOP_K = int(os.getenv("TOP_K", "5"))
MAX_CONTEXT_CHARS = int(os.getenv("MAX_CONTEXT_CHARS", "7000"))
TEMPERATURE = float(os.getenv("TEMPERATURE", "0.2"))
MAX_OUTPUT_TOKENS = int(os.getenv("MAX_OUTPUT_TOKENS", "500"))

print("Default provider:", LLM_PROVIDER)

OPENAI_API_KEY: MISSING
GEMINI_API_KEY: SET
ANTHROPIC_API_KEY: MISSING
HUGGINGFACE_API_KEY: MISSING
Default provider: gemini_model


## 8) Build the Retriever

We use cosine similarity over BERT-mini embeddings to retrieve the most relevant campaigns for a user question.

In [15]:
def retrieve_top_k(query: str, k: int = 5) -> pd.DataFrame:
    query_vec = embed_one_text(query).astype(np.float32, copy=False)
    doc_vecs = embeddings.astype(np.float32, copy=False)

    # Stable cosine similarity in torch to avoid numpy matmul edge warnings.
    q = torch.from_numpy(query_vec)
    d = torch.from_numpy(doc_vecs)
    scores = torch.nn.functional.cosine_similarity(d, q.unsqueeze(0), dim=1, eps=1e-8)
    scores = scores.cpu().numpy()

    top_idx = np.argsort(scores)[::-1][:k]

    out = df.iloc[top_idx].copy()
    out["similarity"] = scores[top_idx]
    return out[["title", "url", "campaign_instance_id", "campaign_text", "similarity"]].reset_index(drop=True)

retrieve_top_k("Which campaigns are about earthquakes?", k=5)

Embedding:   0%|          | 0/1 [00:00<?, ?it/s]

,title,url,campaign_instance_id,campaign_text,similarity
0,Natural disasters in Switzerland: Blatten,https://www.swiss-solidarity.org/campaigns/nat...,natural disasters in switzerland blatten||natu...,Natural disasters in Switzerland: Blatten We n...,0.603506
1,Severe weather events in Switzerland in 2024,https://www.swiss-solidarity.org/campaigns/sev...,severe weather events in switzerland in 2024||...,Severe weather events in Switzerland in 2024 W...,0.602487
2,Jeder Rappen zählt 2009 – 2018,https://www.swiss-solidarity.org/campaigns/jed...,jeder rappen z hlt 2009 2018||jeder-rappen-zae...,Jeder Rappen zählt 2009 – 2018 We need you You...,0.600654
3,Rockslide at Bondo,https://www.swiss-solidarity.org/campaigns/roc...,rockslide at bondo||rockslide-at-bondo__instan...,"Rockslide at Bondo We need you Your donation, ...",0.599283
4,Earthquake in Turkey and Syria,https://www.swiss-solidarity.org/campaigns/ear...,earthquake in turkey and syria||earthquake-in-...,Earthquake in Turkey and Syria We need you You...,0.598308


## 9) Add Multi-Provider LLM Generation

The next cell defines one function per provider (`openai`, `gemini`, `anthropic`, `huggingface`) and a dispatcher so you can switch with one parameter.

In [13]:
def generate_openai(prompt: str, model: str, temperature: float = 0.2, max_tokens: int = 500) -> str:
    from openai import OpenAI

    api_key = API_KEYS["OPENAI_API_KEY"]
    if not api_key:
        raise ValueError("OPENAI_API_KEY is missing.")

    client = OpenAI(api_key=api_key)
    response = client.responses.create(
        model=model,
        input=prompt,
        temperature=temperature,
        max_output_tokens=max_tokens,
    )
    return response.output_text

def generate_gemini(prompt: str, model: str, temperature: float = 0.2, max_tokens: int = 500) -> str:
    import google.generativeai as genai

    api_key = API_KEYS["GEMINI_API_KEY"]
    if not api_key:
        raise ValueError("GEMINI_API_KEY is missing.")

    genai.configure(api_key=api_key)

    # Fallback keeps default as gemma-4-31b-it but improves reliability.
    candidate_models = [model]
    if model != "gemini-1.5-flash":
        candidate_models.append("gemini-1.5-flash")

    last_exc = None
    for candidate_model in candidate_models:
        try:
            gen_model = genai.GenerativeModel(model_name=candidate_model)
            response = gen_model.generate_content(
                prompt,
                generation_config={
                    "temperature": temperature,
                    "max_output_tokens": max_tokens,
                },
            )
            return response.text if hasattr(response, "text") else str(response)
        except Exception as e:
            last_exc = e
            continue

    raise RuntimeError(f"Gemini request failed for models {candidate_models}: {last_exc}")

def generate_anthropic(prompt: str, model: str, temperature: float = 0.2, max_tokens: int = 500) -> str:
    import anthropic

    api_key = API_KEYS["ANTHROPIC_API_KEY"]
    if not api_key:
        raise ValueError("ANTHROPIC_API_KEY is missing.")

    client = anthropic.Anthropic(api_key=api_key)
    message = client.messages.create(
        model=model,
        max_tokens=max_tokens,
        temperature=temperature,
        messages=[{"role": "user", "content": prompt}],
    )
    texts = [block.text for block in message.content if getattr(block, "type", None) == "text"]
    return "\n".join(texts).strip()

def generate_huggingface(prompt: str, model: str, temperature: float = 0.2, max_tokens: int = 500) -> str:
    api_key = API_KEYS["HUGGINGFACE_API_KEY"]
    if not api_key:
        raise ValueError("HUGGINGFACE_API_KEY is missing.")

    url = f"https://api-inference.huggingface.co/models/{model}"
    headers = {"Authorization": f"Bearer {api_key}"}
    payload = {
        "inputs": prompt,
        "parameters": {
            "max_new_tokens": max_tokens,
            "temperature": temperature,
            "return_full_text": False,
        },
    }

    response = requests.post(url, headers=headers, json=payload, timeout=120)
    response.raise_for_status()
    data = response.json()

    if isinstance(data, list) and len(data) > 0 and "generated_text" in data[0]:
        return data[0]["generated_text"].strip()
    return str(data)

def generate_answer(provider: str, prompt: str, temperature: float = None, max_tokens: int = None, model: str = None) -> str:
    provider = provider.strip().lower()
    provider_aliases = {
        "gemini_model": "gemini",
    }
    provider = provider_aliases.get(provider, provider)

    temperature = TEMPERATURE if temperature is None else temperature
    max_tokens = MAX_OUTPUT_TOKENS if max_tokens is None else max_tokens

    if provider == "openai":
        model = model or OPENAI_MODEL
        return generate_openai(prompt, model=model, temperature=temperature, max_tokens=max_tokens)
    if provider == "gemini":
        model = model or GEMINI_MODEL
        return generate_gemini(prompt, model=model, temperature=temperature, max_tokens=max_tokens)
    if provider == "anthropic":
        model = model or ANTHROPIC_MODEL
        return generate_anthropic(prompt, model=model, temperature=temperature, max_tokens=max_tokens)
    if provider == "huggingface":
        model = model or HUGGINGFACE_MODEL
        return generate_huggingface(prompt, model=model, temperature=temperature, max_tokens=max_tokens)

    raise ValueError("Unknown provider. Use one of: openai, gemini, gemini_model, anthropic, huggingface")

## 10) Create the RAG Q&A Function

This function retrieves the top-`k` relevant campaigns, builds a grounded prompt, then asks the selected LLM provider to answer from that context only.

In [10]:
def build_context(retrieved_df: pd.DataFrame, max_chars: int = 7000) -> str:
    chunks = []
    total = 0
    for _, row in retrieved_df.iterrows():
        piece = (
            f"Title: {row['title']}\n"
            f"URL: {row['url']}\n"
            f"Instance ID: {row['campaign_instance_id']}\n"
            f"Similarity: {row['similarity']:.4f}\n"
            f"Text: {row['campaign_text']}\n"
        )
        if total + len(piece) > max_chars:
            break
        chunks.append(piece)
        total += len(piece)
    return "\n---\n".join(chunks)

def build_rag_prompt(question: str, context: str) -> str:
    return f"""You are a data analyst helping swiss solidarity to analyze campaign data.
Answer ONLY from the provided campaign context.
If the answer is not in the context, say so clearly.

Question: {question}

Campaign context:
{context}

Return:
1) A concise answer.
2) Bullet points with the campaign titles and URLs you used as evidence.
"""

def ask_rag(
    question: str,
    provider: str = None,
    k: int = None,
    model: str = None,
    temperature: float = None,
    max_tokens: int = None,
    max_context_chars: int = None,
):
    provider = (provider or LLM_PROVIDER).lower()
    k = TOP_K if k is None else k
    max_context_chars = MAX_CONTEXT_CHARS if max_context_chars is None else max_context_chars

    retrieved = retrieve_top_k(question, k=k)
    context = build_context(retrieved, max_chars=max_context_chars)
    prompt = build_rag_prompt(question, context)

    answer = generate_answer(
        provider=provider,
        prompt=prompt,
        model=model,
        temperature=temperature,
        max_tokens=max_tokens,
    )

    return {
        "provider": provider,
        "question": question,
        "answer": answer,
        "retrieved": retrieved,
        "prompt": prompt,
    }

## 11) Ask Questions (Provider Can Be Switched)

Set `provider` to one of `openai`, `gemini`, `anthropic`, or `huggingface`. If omitted, the notebook uses `LLM_PROVIDER` from environment variables.

In [22]:
question = "Which campaigns in the dataset are related to earthquakes, and where did they occur?"

# Change provider here: openai | gemini | anthropic | huggingface
# Make the RAG call robust if cells were run out of order.
required_cols = {"campaign_instance_id", "campaign_text"}
missing = required_cols - set(df.columns)

if missing:
    # Rebuild campaign_instance_id if needed
    if "campaign_instance_id" not in df.columns:
        slug = df["url"].fillna("").str.rstrip("/").str.split("/").str[-1]
        slug_base = slug.str.replace(r"-\d+$", "", regex=True)
        title_norm = (
            df["title"].fillna("")
            .str.lower()
            .str.replace(r"[^a-z0-9\s]+", " ", regex=True)
            .str.replace(r"\s+", " ", regex=True)
            .str.strip()
        )
        df["campaign_family_key"] = title_norm + "||" + slug_base
        df = df.sort_values(["campaign_family_key", "url"]).reset_index(drop=True)
        df["instance_number"] = df.groupby("campaign_family_key").cumcount() + 1
        df["campaign_instance_id"] = df["campaign_family_key"] + "__instance_" + df["instance_number"].astype(str)

    # Rebuild campaign_text if needed
    if "campaign_text" not in df.columns:
        df["campaign_text"] = df.apply(combine_text, axis=1)

result = ask_rag(question=question, provider=LLM_PROVIDER, k=5)

#print("Provider:", result["provider"])
#print("\nAnswer:\n")
print(result["answer"])

print("\nTop retrieved evidence rows:\n")
result["retrieved"][["title", "url", "similarity"]]

Embedding:   0%|          | 0/1 [00:00<?, ?it/s]

*   Role: Data analyst for Swiss Solidarity.
    *   Task: Identify campaigns related to earthquakes and their locations.
    *   Constraint: Answer ONLY from the provided context. If not found, say so clearly.
    *   Output Format: 1) Concise answer, 2) Bullet points with titles and URLs.

    *   Campaign 1: "Severe weather events in Switzerland in 2024" -> Severe weather, Switzerland.
    *   Campaign 2: "Natural disasters in Switzerland: Blatten" -> Natural disasters, Blatten (Switzerland).
    *   Campaign 3: "Aid in response to disasters and crises in Switzerland" -> Disasters/crises, Switzerland.
    *   Campaign 4: "Aid following the earthquake in Nepal" -> Earthquake, Nepal.
    *   Campaign 5: "Jeder Rappen zählt 2009 – 2018" -> General fundraising.

    *   Only one campaign mentions an earthquake: "Aid following the earthquake in Nepal".
    *   Location: Nepal.

    *   Concise answer: The campaign related to earthquakes is "Aid following the earthquake in Nepal," which o

,title,url,similarity
0,Severe weather events in Switzerland in 2024,https://www.swiss-solidarity.org/campaigns/sev...,0.644430
1,Natural disasters in Switzerland: Blatten,https://www.swiss-solidarity.org/campaigns/nat...,0.642399
2,Aid in response to disasters and crises in Swi...,https://www.swiss-solidarity.org/campaigns/aid...,0.639838
3,Aid following the earthquake in Nepal,https://www.swiss-solidarity.org/campaigns/aid...,0.639647
4,Jeder Rappen zählt 2009 – 2018,https://www.swiss-solidarity.org/campaigns/jed...,0.639348
